In [1]:
import sys
print(sys.executable)

c:\Users\AJYH\AppData\Local\Programs\Python\Python38\python.exe


# Tutorial 1: <br><br>Echo State Networks for Text Classification

This notebook presents a use case example of the ``EsnTorch`` library. It describes the implementation of an **Echo State Network (ESN)** for text classification on the **TREC-6** dataset (question classification).

The instantiation, training and evaluation of an ESN for text classification is achieved via the following steps:
1. Import libraries and modules
2. Load and prepare data
3. Instantiate the model:
    - specify parameters
    - specify learning algorithm
    - warm up
4. Train
5. Evaluate

## Librairies

In [1]:
# !pip install transformers==4.8.2
# !pip install datasets==1.7.0
# !pip install tqdm
# !pip install torch==1.7.1+cu110 torchvision==0.8.2+cu110 torchaudio==0.7.2 -f https://download.pytorch.org/whl/torch_stable.html

In [2]:
# To enable progress bars in jupyter:

# pip install ipywidgets
# jupyter nbextension enable --py widgetsnbextension
# conda install -c conda-forge nodejs
# jupyter labextension install @jupyter-widgets/jupyterlab-manager

In [3]:
# Comment this if library is installed!
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

In [4]:
from tqdm.notebook import tqdm_notebook
from sklearn.metrics import classification_report

import torch

from datasets import load_dataset, Dataset, concatenate_datasets

from transformers import AutoTokenizer
from transformers.data.data_collator import DataCollatorWithPadding

import esntorch.core.reservoir as res
import esntorch.core.learning_algo as la
import esntorch.core.pooling_strategy as ps
import esntorch.core.esn as esn

In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
# Set device (cpu or gpu if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

## Load and prepare data

### Load and tokenize data

In [7]:
# Custom functions for loading and preparing data
def tokenize(sample):
    """Tokenize sample"""
    
    sample = tokenizer(sample['text'], truncation=True, padding=False, return_length=True)
    
    return sample
    
def load_and_prepare_dataset(dataset_name, split, cache_dir):
    """
    Load dataset from the datasets library of HuggingFace.
    Tokenize and add length.
    """
    
    # Load dataset
    dataset = load_dataset(dataset_name, split=split, cache_dir=CACHE_DIR)
    
    # Rename label column for tokenization purposes (use 'label-fine' for fine-grained labels)
    dataset = dataset.rename_column('label-coarse', 'labels')
    
    # Tokenize data
    dataset = dataset.map(tokenize, batched=True)
    dataset = dataset.rename_column('length', 'lengths')
    dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels', 'lengths'])
    
    return dataset

In [8]:
# Load BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Load and prepare data
CACHE_DIR = 'cache_dir/' # put your path here

full_dataset = load_and_prepare_dataset('trec', split=None, cache_dir=CACHE_DIR)
train_dataset = full_dataset['train'].sort("lengths")
test_dataset = full_dataset['test'].sort("lengths")

# Create dict of all datasets
dataset_d = {
    'train': train_dataset,
    'test': test_dataset
    }

Using the latest cached version of the module from C:\Users\AJYH\.cache\huggingface\modules\datasets_modules\datasets\trec\751da1ab101b8d297a3d6e9c79ee9b0173ff94c4497b75677b59b61d5467a9b9 (last modified on Tue Mar  3 11:36:19 2026) since it couldn't be found locally at trec, or remotely on the Hugging Face Hub.


Generating train split:   0%|          | 0/5452 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/5452 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [9]:
dataset_d

{'train': Dataset({
     features: ['labels', 'label-fine', 'text', 'input_ids', 'token_type_ids', 'attention_mask', 'lengths'],
     num_rows: 5452
 }),
 'test': Dataset({
     features: ['labels', 'label-fine', 'text', 'input_ids', 'token_type_ids', 'attention_mask', 'lengths'],
     num_rows: 500
 })}

### Create dataloaders

In [10]:
# Create dict of dataloaders

dataloader_d = {}

for k, v in dataset_d.items():
    dataloader_d[k] = torch.utils.data.DataLoader(v, batch_size=256, 
                                                  collate_fn=DataCollatorWithPadding(tokenizer))

In [11]:
dataloader_d

{'train': <torch.utils.data.dataloader.DataLoader at 0x1de280ad4c0>,
 'test': <torch.utils.data.dataloader.DataLoader at 0x1de280ad4f0>}

## Instanciate the model

### Parameters

Most parameters are self-explanatory for a reader familiar with ESNs. Please refer to the documentation for further details. 

The ``mode`` parameter represents the type of reservoir to be considered:
- ``mode = recurrent_layer``: implements a **classical recurrent reservoir**, specified among others by its ``dim``, ``sparsity``, ``spectral_radius``, ``leaking_rate`` and ``activation_function``.
- ``mode = linear_layer``: implements a simple **linear layer** specified by its ``dim`` and ``activation_function``.
- ``mode = no_layer``: implements **the absence of reservoir**, meaning that the embedded inputs are directly fed to the the learning algorithms.

The comparison between the ``recurrent_layer`` and the ``no_layer``modes enables to assess the impact of the reservoir on the results. The comparison between the ``recurrent_layer`` and the ``linear_layer``modes allows to assess the importance of the recurrence of the reservoir on the results.

In [12]:
# ESN parameters
esn_params = {
            'embedding': 'bert-base-uncased', # name of Hugging Face model
            'dim': 1000,
            'sparsity': 0.9,
            'spectral_radius': 0.9,
            'leaking_rate': 0.5,
            'activation_function': 'tanh', # 'tanh', 'relu'
            'bias_scaling': 0.1,
            'input_scaling': 0.1,
            'mean': 0.0,
            'std': 1.0,     
            'learning_algo': None,         # initialzed below
            'criterion': None,             # initialzed below (only for learning algos trained with SGD)
            'optimizer': None,             # initialzed below (only for learning algos trained with SGD)
            'pooling_strategy': 'mean',    # 'mean', 'last', None
            'bidirectional': False,        # True, False
            'mode' : 'recurrent_layer',    # 'no_layer', 'linear_layer', 'recurrent_layer'
            'device': device,  
            'seed': 42
            }

# Instantiate the ESN
ESN = esn.EchoStateNetwork(**esn_params)

Model downloaded: bert-base-uncased


### Learning algorithm
Choose your learning algo by un-commenting its associated cell.

The following algorithms are trained via a **direct method**.<br>
Hence, there is no need to specify any *criterion* and *optimizer*.
- ``RidgeRegression`` (our implementation)
- ``RidgeRegression_skl`` (from scikit-learn)
- ``LinearSVC`` (from scikit-learn)
- ``LogisticRegression_skl`` (from scikit-learn)

In [13]:
ESN.learning_algo = la.RidgeRegression(alpha=10.0)

In [14]:
# ESN.learning_algo = la.RidgeRegression_skl(alpha=10.0)

In [15]:
# ESN.learning_algo = la.LinearSVC(C=1.0)

In [16]:
# ESN.learning_algo = la.LogisticRegression_skl()

The following algorithms are trained via a **gradient descent**.<br>
Accordingly, a *criterion* and an *optimizer* must be specified.
- ``LogisticRegression`` (our implementation)
- ``DeepNN`` (our implementation)

In [17]:
# if esn_params['mode'] == 'no_layer':
#     input_dim = ESN.layer.input_dim
# else:
#     input_dim = ESN.layer.dim

# if esn_params['bidirectional']:
#     input_dim *= 2 

In [18]:
# ESN.learning_algo = la.LogisticRegression(input_dim=input_dim, output_dim=6)

In [19]:
# ESN.learning_algo = la.DeepNN([input_dim, 512, 256, 6])

In [20]:
# # Needs criterion and otpimizer

# ESN.criterion = torch.nn.CrossEntropyLoss()
# ESN.optimizer = torch.optim.Adam(ESN.learning_algo.parameters(), lr=0.01)

### Warm up

In [21]:
# Put model on device
ESN = ESN.to(device)

In [22]:
# Warm up the ESN on multiple sentences
if isinstance(ESN.layer, res.LayerRecurrent):
    ESN.warm_up(dataset_d['train'].select(range(10)))

## Train

For **direct methods**, the parameters ``epochs`` and ``iter_steps`` are ignored. 

In [23]:
ESN.fit(dataloader_d["train"], epochs=3, iter_steps=50)

Computing closed-form solution...


  0%|          | 0/22 [00:00<?, ?it/s]


Training complete.


## Evaluate

In [24]:
# Train predictions and accuracy
train_pred, train_acc = ESN.predict(dataloader_d["train"], verbose=False)
train_acc

  0%|          | 0/22 [00:00<?, ?it/s]

93.34189288334557

In [25]:
# Test predictions
test_pred, test_acc = ESN.predict(dataloader_d["test"], verbose=False)
test_acc

  0%|          | 0/2 [00:00<?, ?it/s]

92.0

In [26]:
# Test classification report
print(classification_report(test_pred.tolist(), 
                            dataset_d['test']['labels'].tolist(), 
                            digits=4))

              precision    recall  f1-score   support

           0     0.9638    0.8808    0.9204       151
           1     0.7766    0.8902    0.8295        82
           2     0.7778    1.0000    0.8750         7
           3     0.9692    0.9692    0.9692        65
           4     0.9735    0.9649    0.9692       114
           5     0.9136    0.9136    0.9136        81

    accuracy                         0.9200       500
   macro avg     0.8957    0.9365    0.9128       500
weighted avg     0.9253    0.9200    0.9212       500

